In [ ]:
import itertools
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns
from tqdm import tqdm

out_dir = Path("out")
out_dir.mkdir(exist_ok=True, parents=True)

pl.Config.set_tbl_rows(100)

train_df = pl.read_csv("../data/raw/train.csv")
test_df = pl.read_csv("../data/raw/test.csv")
submit_df = pl.read_csv("../data/raw/sample_submission.csv")

train_df.head()

In [ ]:
train_df = train_df.drop("id").cast({"date": pl.Date})
test_df = test_df.drop("id").cast({"date": pl.Date})

train_df.describe()

In [ ]:
(
    train_df.group_by("country", "store", "product")
    .agg(pl.col("num_sold").null_count())
    .filter(pl.col("num_sold") > 0)
)

In [30]:
countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()

out_dir = Path("./out/num_sold_density")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        train_df.drop_nulls().group_by("date", category).agg(pl.col("num_sold").mean())
    )
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.histplot(tmp, x="num_sold", hue=category, kde=True, alpha=0.2, ax=ax)
    ax.set_title(f"num_sold density by {category}")
    ax.set_xlabel("num_sold")
    ax.set_ylabel("density")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)


In [31]:
out_dir = Path("./out/num_sold_over_date")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        train_df.drop_nulls().group_by("date", category).agg(pl.col("num_sold").mean())
    )
    fig, ax = plt.subplots(figsize=(20, 8))
    sns.lineplot(data=tmp, x="date", y="num_sold", hue=category, ax=ax)
    ax.set_title(f"num_sold over date by {category}")
    ax.set_xlabel("date")
    ax.set_ylabel("num_sold")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)

In [58]:
from typing import cast

from polars_ml import Pipeline
from polars_ml.preprocess import MinMaxScaler

pp = (
    Pipeline()
    .min_max_scale("num_sold", by=["country", "store"], component_name="min_max")
    .display()
    .inverse_scale("min_max", {"num_sold": "num_sold"})
    .display()
)

statistic,date,country,store,product,num_sold,num_sold_original,diff
str,str,str,str,str,f64,f64,f64
"""count""","""230130""","""230130""","""230130""","""230130""",221259.0,221259.0,221259.0
"""null_count""","""0""","""0""","""0""","""0""",8871.0,8871.0,8871.0
"""mean""","""2013-07-02 00:00:00""",null,null,null,752.527382,752.527382,7.3205e-16
"""std""",null,null,null,null,690.165445,690.165445,8.9633e-15
"""min""","""2010-01-01""","""Canada""","""Discount Stickers""","""Holographic Goose""",5.0,5.0,0.0
"""25%""","""2011-10-02""",null,null,null,219.0,219.0,0.0
"""50%""","""2013-07-02""",null,null,null,605.0,605.0,0.0
"""75%""","""2015-04-02""",null,null,null,1114.0,1114.0,0.0
"""max""","""2016-12-31""","""Singapore""","""Stickers for Less""","""Kerneler Dark Mode""",5939.0,5939.0,9.0949e-13
